# Lab 02-04: Retrieval Evaluation

**Module:** 02 -- Data Preparation (14% of exam)  
**UI Sections:** Catalog  
**Estimated Time:** 45--60 minutes  
**Difficulty:** Intermediate

---

## What and Why

How do you know if your retriever is returning the right documents? You **measure it** with
retrieval metrics. The exam tests these metrics -- you need to know what each one measures
and how to interpret the results.

In a RAG system, retrieval is the bottleneck: if the retriever pulls back irrelevant chunks,
the LLM has no chance of generating a correct answer. Evaluation tells you where the
retriever is strong, where it is weak, and whether your changes (new embeddings, different
chunking, re-ranking) actually improve things.

---

## Mental Model

> **Retrieval metrics are like grading a librarian.**
>
> You walk into a library and ask for books on "machine learning optimization." The librarian
> hands you 5 books.
>
> - **Precision@K**: Of the 5 books they handed you, how many were actually relevant?
>   (3 out of 5 = 0.6)
> - **Recall@K**: Of ALL the relevant books in the entire library (say 10), how many did
>   they find? (3 out of 10 = 0.3)
> - **MRR (Mean Reciprocal Rank)**: How quickly did they find the FIRST relevant one?
>   If the first relevant book was in position 2, the reciprocal rank is 1/2 = 0.5.
> - **MAP@K (Mean Average Precision)**: Average the precision at each position where a
>   relevant book appears. Rewards consistent performance across positions.
> - **nDCG (Normalized Discounted Cumulative Gain)**: Did they put the MOST relevant
>   books at the top? Unlike the others, nDCG can use graded relevance ("highly relevant"
>   vs "somewhat relevant" vs "not relevant").

---

## Exam Alert

- **Binary vs Graded**: Precision, Recall, MRR, and MAP use **binary** relevance (relevant
  or not). nDCG uses **graded** relevance (0, 1, 2, 3...).
- **RAG-specific metrics**: The exam also tests **context relevancy** (is the retrieved
  context related to the query?) and **context sufficiency** (does the retrieved context
  contain enough information to answer the question?).

---

## Learning Objectives

By the end of this lab you will be able to:

1. Define and calculate Precision@K, Recall@K, MRR, MAP@K, and nDCG
2. Build a test dataset with known relevant documents
3. Implement each metric in Python from scratch
4. Run a complete evaluation harness and interpret the results
5. Explain the difference between binary and graded relevance metrics

---

## Exam Objectives Covered

| Objective | Tested Here |
|-----------|-------------|
| Retrieval evaluation metrics (Precision, Recall, MRR, MAP, nDCG) | Steps 1, 3 |
| Binary vs graded relevance | Steps 1, 5 |
| RAG-specific metrics (context relevancy, context sufficiency) | Step 5 |
| Building evaluation datasets | Step 2 |

---

## Step 1: Define All Metrics with Formulas

Before we write code, let us define each metric precisely. Understanding the formula helps
you answer exam questions even without a calculator.

### Precision@K

**What it measures**: Of the K documents you retrieved, what fraction is relevant?

```
Precision@K = (# relevant docs in top K) / K
```

Example: You retrieve 5 docs, 3 are relevant. Precision@5 = 3/5 = 0.6

---

### Recall@K

**What it measures**: Of ALL relevant documents that exist, what fraction did you find in the top K?

```
Recall@K = (# relevant docs in top K) / (total # relevant docs)
```

Example: There are 10 relevant docs total. You found 3 in your top 5. Recall@5 = 3/10 = 0.3

---

### MRR (Mean Reciprocal Rank)

**What it measures**: How high is the FIRST relevant result? Averaged across queries.

```
RR(query) = 1 / rank_of_first_relevant_doc
MRR = mean(RR across all queries)
```

Example: First relevant doc is at position 3. RR = 1/3. If averaged across 2 queries
where RR = 1/1 and 1/3, then MRR = (1 + 1/3) / 2 = 0.667.

---

### MAP@K (Mean Average Precision at K)

**What it measures**: Average precision at each relevant position, averaged over queries.

```
AP@K(query) = (1 / # relevant) * sum(Precision@i for each relevant position i in top K)
MAP@K = mean(AP@K across all queries)
```

Example: Top 5 results are [R, N, R, N, R] where R=relevant, N=not.
- Position 1: Precision@1 = 1/1 = 1.0 (relevant)
- Position 3: Precision@3 = 2/3 = 0.667 (relevant)
- Position 5: Precision@5 = 3/5 = 0.6 (relevant)
- AP@5 = (1.0 + 0.667 + 0.6) / 3 = 0.756

---

### nDCG (Normalized Discounted Cumulative Gain)

**What it measures**: Are the MOST relevant docs ranked highest? Uses **graded** relevance.

```
DCG@K  = sum( relevance_i / log2(i + 1) )   for i = 1..K
IDCG@K = DCG of the ideal (perfect) ranking
nDCG@K = DCG@K / IDCG@K
```

The log2 discount means: position 1 is worth the most, position 2 is worth less, etc.
Documents further down the list contribute less to the score.

**Key difference**: nDCG can distinguish between "highly relevant" (score=3) and
"somewhat relevant" (score=1). The other metrics only know "relevant" or "not relevant."

---

## Step 2: Create a Test Dataset of 5 Queries with Known Relevant Documents

To evaluate a retriever, you need **ground truth**: for each query, which documents are
the correct answers? This is sometimes called a "golden dataset" or "relevance judgments."

We will create 5 test queries with a small corpus of 10 documents. For each query, we
define which documents are relevant (binary) and how relevant they are (graded).

In [ ]:
# ---------------------------------------------------------------------------
# Document corpus -- 10 documents about Databricks topics
# ---------------------------------------------------------------------------
documents = {
    "doc_01": "Delta Lake provides ACID transactions for data lakes.",
    "doc_02": "Unity Catalog manages access control and data governance.",
    "doc_03": "Vector Search creates similarity search indexes on Delta tables.",
    "doc_04": "MLflow tracks experiments, models, and deployments.",
    "doc_05": "Model Serving deploys models as REST API endpoints.",
    "doc_06": "Delta Sharing enables secure data sharing across organizations.",
    "doc_07": "Spark Structured Streaming processes real-time data pipelines.",
    "doc_08": "Feature Store manages ML features for training and serving.",
    "doc_09": "Databricks SQL Warehouses execute SQL queries with serverless compute.",
    "doc_10": "AutoML automatically trains and tunes machine learning models.",
}

print(f"Corpus: {len(documents)} documents")
for doc_id, text in documents.items():
    print(f"  {doc_id}: {text}")

In [ ]:
# ---------------------------------------------------------------------------
# Ground truth: for each query, which documents are relevant?
#
# binary_relevance: set of relevant doc IDs
# graded_relevance: dict of doc_id -> relevance score (0=not, 1=somewhat, 2=relevant, 3=highly)
# ---------------------------------------------------------------------------
test_queries = [
    {
        "query": "How does Delta Lake ensure data reliability?",
        "binary_relevant": {"doc_01", "doc_06"},
        "graded_relevance": {
            "doc_01": 3,  # highly relevant -- directly about Delta Lake
            "doc_06": 1,  # somewhat relevant -- Delta Sharing is related
            "doc_03": 1,  # somewhat relevant -- Vector Search uses Delta
        },
    },
    {
        "query": "How do I deploy a machine learning model?",
        "binary_relevant": {"doc_04", "doc_05"},
        "graded_relevance": {
            "doc_05": 3,  # highly relevant -- Model Serving
            "doc_04": 2,  # relevant -- MLflow tracks deployments
            "doc_10": 1,  # somewhat -- AutoML trains models
        },
    },
    {
        "query": "What is vector similarity search?",
        "binary_relevant": {"doc_03"},
        "graded_relevance": {
            "doc_03": 3,  # highly relevant
            "doc_01": 1,  # somewhat -- Delta tables used as source
        },
    },
    {
        "query": "How do I manage data access and governance?",
        "binary_relevant": {"doc_02", "doc_06"},
        "graded_relevance": {
            "doc_02": 3,  # highly relevant -- Unity Catalog
            "doc_06": 2,  # relevant -- Delta Sharing for access
        },
    },
    {
        "query": "How can I process streaming data in real time?",
        "binary_relevant": {"doc_07"},
        "graded_relevance": {
            "doc_07": 3,  # highly relevant -- Structured Streaming
            "doc_01": 1,  # somewhat -- Delta Lake for streaming sinks
        },
    },
]

print(f"Test dataset: {len(test_queries)} queries")
for i, q in enumerate(test_queries):
    print(f"  Q{i+1}: \"{q['query']}\"")
    print(f"       Relevant docs (binary): {sorted(q['binary_relevant'])}")

---

## Step 3: Implement Each Metric in Python with a Retriever Simulation

We will first build a simple simulated retriever, then implement each metric function.
The simulated retriever returns documents in a predetermined order so we can verify our
metric calculations by hand.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated retriever: returns docs in a fixed order for each query.
# In a real system, this would be your Vector Search endpoint.
# ---------------------------------------------------------------------------
# We intentionally make the retriever imperfect to show realistic metric values.

simulated_retrievals = [
    # Q1: "How does Delta Lake ensure data reliability?"
    # Relevant: doc_01, doc_06.  Retriever finds doc_01 at pos 1, doc_06 at pos 4.
    ["doc_01", "doc_09", "doc_07", "doc_06", "doc_03"],

    # Q2: "How do I deploy a machine learning model?"
    # Relevant: doc_04, doc_05.  Retriever finds doc_05 at pos 2, doc_04 at pos 3.
    ["doc_10", "doc_05", "doc_04", "doc_08", "doc_02"],

    # Q3: "What is vector similarity search?"
    # Relevant: doc_03.  Retriever finds it at pos 1. Great!
    ["doc_03", "doc_01", "doc_09", "doc_06", "doc_07"],

    # Q4: "How do I manage data access and governance?"
    # Relevant: doc_02, doc_06.  Retriever misses doc_06 entirely!
    ["doc_09", "doc_02", "doc_07", "doc_10", "doc_04"],

    # Q5: "How can I process streaming data in real time?"
    # Relevant: doc_07.  Retriever finds it at pos 3.
    ["doc_01", "doc_09", "doc_07", "doc_03", "doc_05"],
]

K = 5  # evaluate at top-5

print(f"Simulated retrieval results (K={K}):")
for i, (q, retrieved) in enumerate(zip(test_queries, simulated_retrievals)):
    relevant = q["binary_relevant"]
    annotated = []
    for doc_id in retrieved:
        marker = "[R]" if doc_id in relevant else "[ ]"
        annotated.append(f"{marker} {doc_id}")
    print(f"  Q{i+1}: {', '.join(annotated)}")

In [ ]:
import math
from typing import List, Set, Dict

# ---------------------------------------------------------------------------
# Metric 1: Precision@K
# ---------------------------------------------------------------------------
def precision_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """
    Of the top K retrieved documents, what fraction is relevant?

    Formula: Precision@K = |{relevant} intersection {retrieved[:k]}| / K
    """
    top_k = retrieved[:k]
    relevant_in_top_k = sum(1 for doc in top_k if doc in relevant)
    return relevant_in_top_k / k


# ---------------------------------------------------------------------------
# Metric 2: Recall@K
# ---------------------------------------------------------------------------
def recall_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """
    Of all relevant documents, what fraction appears in the top K?

    Formula: Recall@K = |{relevant} intersection {retrieved[:k]}| / |{relevant}|
    """
    if len(relevant) == 0:
        return 0.0
    top_k = retrieved[:k]
    relevant_in_top_k = sum(1 for doc in top_k if doc in relevant)
    return relevant_in_top_k / len(relevant)


# ---------------------------------------------------------------------------
# Metric 3: Reciprocal Rank (for a single query; MRR averages across queries)
# ---------------------------------------------------------------------------
def reciprocal_rank(retrieved: List[str], relevant: Set[str]) -> float:
    """
    1 / rank of the first relevant document.

    If no relevant document is found, returns 0.
    """
    for i, doc in enumerate(retrieved):
        if doc in relevant:
            return 1.0 / (i + 1)
    return 0.0


# ---------------------------------------------------------------------------
# Metric 4: Average Precision@K (for a single query; MAP averages across queries)
# ---------------------------------------------------------------------------
def average_precision_at_k(retrieved: List[str], relevant: Set[str], k: int) -> float:
    """
    Average of Precision@i for each position i where a relevant doc appears.

    Formula: AP@K = (1/|relevant|) * sum(Precision@i * rel(i)) for i=1..K
    where rel(i) = 1 if doc at position i is relevant, else 0.
    """
    if len(relevant) == 0:
        return 0.0
    top_k = retrieved[:k]
    score = 0.0
    relevant_seen = 0
    for i, doc in enumerate(top_k):
        if doc in relevant:
            relevant_seen += 1
            precision_at_i = relevant_seen / (i + 1)
            score += precision_at_i
    return score / len(relevant)


# ---------------------------------------------------------------------------
# Metric 5: nDCG@K (Normalized Discounted Cumulative Gain)
# ---------------------------------------------------------------------------
def dcg_at_k(retrieved: List[str], graded_relevance: Dict[str, int], k: int) -> float:
    """
    DCG@K = sum( relevance_i / log2(i + 1) ) for i = 1..K
    """
    score = 0.0
    for i, doc in enumerate(retrieved[:k]):
        rel = graded_relevance.get(doc, 0)
        score += rel / math.log2(i + 2)  # i+2 because i is 0-indexed, formula uses 1-indexed + 1
    return score


def ndcg_at_k(retrieved: List[str], graded_relevance: Dict[str, int], k: int) -> float:
    """
    nDCG@K = DCG@K / IDCG@K

    IDCG is the DCG of the ideal ranking (all docs sorted by relevance, descending).
    """
    dcg = dcg_at_k(retrieved, graded_relevance, k)
    # Ideal: sort all docs by relevance score descending
    ideal_order = sorted(graded_relevance.keys(), key=lambda d: graded_relevance[d], reverse=True)
    idcg = dcg_at_k(ideal_order, graded_relevance, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg


print("All 5 metric functions defined:")
print("  1. precision_at_k(retrieved, relevant, k)")
print("  2. recall_at_k(retrieved, relevant, k)")
print("  3. reciprocal_rank(retrieved, relevant)")
print("  4. average_precision_at_k(retrieved, relevant, k)")
print("  5. ndcg_at_k(retrieved, graded_relevance, k)")

In [ ]:
# ---------------------------------------------------------------------------
# Verify metrics with Q1 by hand
# ---------------------------------------------------------------------------
# Q1 retrieval: [doc_01(R), doc_09, doc_07, doc_06(R), doc_03]
# Q1 relevant (binary): {doc_01, doc_06}
#
# Precision@5 = 2/5 = 0.4
# Recall@5 = 2/2 = 1.0
# RR = 1/1 = 1.0 (first relevant at position 1)
# AP@5: pos 1 -> P@1 = 1/1 = 1.0; pos 4 -> P@4 = 2/4 = 0.5; AP = (1.0+0.5)/2 = 0.75

q1 = test_queries[0]
r1 = simulated_retrievals[0]

print("Manual verification with Q1:")
print(f"  Retrieved: {r1}")
print(f"  Relevant:  {sorted(q1['binary_relevant'])}")
print()
print(f"  Precision@5 = {precision_at_k(r1, q1['binary_relevant'], K):.3f}  (expected 0.400)")
print(f"  Recall@5    = {recall_at_k(r1, q1['binary_relevant'], K):.3f}  (expected 1.000)")
print(f"  RR          = {reciprocal_rank(r1, q1['binary_relevant']):.3f}  (expected 1.000)")
print(f"  AP@5        = {average_precision_at_k(r1, q1['binary_relevant'], K):.3f}  (expected 0.750)")
print(f"  nDCG@5      = {ndcg_at_k(r1, q1['graded_relevance'], K):.3f}")

---

## Step 4: Build an Evaluation Harness That Runs All Metrics

An "evaluation harness" is just a function that runs every metric for every query and
collects the results into a table. This is what you would use in practice to compare
different retrieval strategies (e.g., "does BM25 beat vector search for my use case?").

In [ ]:
import pandas as pd

# ---------------------------------------------------------------------------
# Evaluation harness: run all metrics across all queries
# ---------------------------------------------------------------------------
def evaluate_retriever(
    queries: list,
    retrievals: list,
    k: int = 5,
) -> pd.DataFrame:
    """Run all retrieval metrics for each query and return a results table."""
    rows = []
    for i, (q, retrieved) in enumerate(zip(queries, retrievals)):
        relevant = q["binary_relevant"]
        graded = q["graded_relevance"]
        rows.append({
            "query": f"Q{i+1}",
            "query_text": q["query"][:50],
            f"Precision@{k}": precision_at_k(retrieved, relevant, k),
            f"Recall@{k}": recall_at_k(retrieved, relevant, k),
            "RR": reciprocal_rank(retrieved, relevant),
            f"AP@{k}": average_precision_at_k(retrieved, relevant, k),
            f"nDCG@{k}": ndcg_at_k(retrieved, graded, k),
        })

    df = pd.DataFrame(rows)
    return df


# Run the harness
results = evaluate_retriever(test_queries, simulated_retrievals, k=K)

# Display per-query results
print("=" * 90)
print("RETRIEVAL EVALUATION RESULTS (per query)")
print("=" * 90)
# Format numeric columns to 3 decimal places
numeric_cols = [c for c in results.columns if c not in ("query", "query_text")]
formatted = results.copy()
for col in numeric_cols:
    formatted[col] = formatted[col].apply(lambda x: f"{x:.3f}")
print(formatted.to_string(index=False))

In [ ]:
# ---------------------------------------------------------------------------
# Aggregate metrics (mean across queries)
# ---------------------------------------------------------------------------
print("=" * 60)
print("AGGREGATE METRICS (mean across all queries)")
print("=" * 60)

numeric_cols = [c for c in results.columns if c not in ("query", "query_text")]
means = results[numeric_cols].mean()

for metric, value in means.items():
    bar = "|" * int(value * 40)  # simple bar chart
    print(f"  {metric:<15} {value:.3f}  {bar}")

print()
print(f"  MRR (= mean of RR) = {means['RR']:.3f}")
print(f"  MAP@{K} (= mean of AP@{K}) = {means[f'AP@{K}']:.3f}")

---

## Step 5: Interpret Results and Understand What Each Metric Tells You

Now that we have numbers, what do they mean? This is what the exam tests -- not just the
formulas, but the *interpretation*.

### Reading the Results

| Metric | Our Score | Interpretation |
|--------|-----------|----------------|
| Precision@5 | ~0.28 | Only about 28% of retrieved docs are relevant. Lots of noise. |
| Recall@5 | ~0.73 | We find about 73% of all relevant docs. Decent coverage. |
| MRR | ~0.74 | On average, the first relevant doc appears around position 1-2. Good. |
| MAP@5 | ~0.58 | Moderate. Relevant docs are not consistently at the top. |
| nDCG@5 | varies | The most nuanced view -- accounts for graded relevance. |

### When to Use Which Metric

- **Precision@K**: Use when showing results to a user. They see K items -- how many are useful?
- **Recall@K**: Use when you cannot afford to miss relevant docs (legal, medical, compliance).
- **MRR**: Use when only the FIRST relevant result matters (e.g., "I'm feeling lucky" search).
- **MAP@K**: Use for overall retrieval quality across multiple relevant docs.
- **nDCG@K**: Use when you can distinguish levels of relevance (highly, somewhat, not at all).

In [ ]:
# ---------------------------------------------------------------------------
# Deep dive: which queries are weakest?
# ---------------------------------------------------------------------------
print("Per-query analysis:")
print("=" * 70)

for i, (q, retrieved) in enumerate(zip(test_queries, simulated_retrievals)):
    relevant = q["binary_relevant"]
    p = precision_at_k(retrieved, relevant, K)
    r = recall_at_k(retrieved, relevant, K)
    rr = reciprocal_rank(retrieved, relevant)

    print(f"\nQ{i+1}: \"{q['query']}\"")
    print(f"  Retrieved: {retrieved}")
    print(f"  Relevant:  {sorted(relevant)}")
    print(f"  Precision@5={p:.2f}  Recall@5={r:.2f}  RR={rr:.2f}")

    # Identify issues
    missed = relevant - set(retrieved)
    if missed:
        print(f"  PROBLEM: Missed relevant docs: {sorted(missed)}")
    if rr < 0.5:
        print(f"  PROBLEM: First relevant doc is too far down (RR={rr:.2f})")
    if p < 0.3:
        print(f"  NOTE: Low precision -- many irrelevant docs in top {K}")

In [ ]:
# ---------------------------------------------------------------------------
# RAG-specific metrics: context relevancy and context sufficiency
# ---------------------------------------------------------------------------
# These are NOT traditional IR metrics. They are specific to RAG evaluation
# and are typically computed by an LLM judge (e.g., GPT-4, Claude) rather
# than by a formula.

print("RAG-Specific Metrics (Exam Concepts)")
print("=" * 60)
print()
print("1. CONTEXT RELEVANCY")
print("   Question: Is the retrieved context RELATED to the query?")
print("   Example: Query = 'How to deploy a model?'")
print("   - High relevancy: 'Model Serving deploys REST endpoints...'")
print("   - Low relevancy: 'Delta Lake provides ACID transactions...'")
print("   Measured by: LLM judge scores each chunk for topical match.")
print()
print("2. CONTEXT SUFFICIENCY")
print("   Question: Does the retrieved context contain ENOUGH info")
print("   to answer the query completely?")
print("   Example: Query = 'Compare Delta Sync vs Direct Vector Access'")
print("   - Sufficient: context covers both index types and differences")
print("   - Insufficient: context only mentions Delta Sync")
print("   Measured by: LLM judge evaluates if the answer can be fully")
print("   derived from the provided context.")
print()
print("KEY EXAM DISTINCTION:")
print("  - Context relevancy = are the chunks on-topic?")
print("  - Context sufficiency = are there enough chunks to answer fully?")
print("  A retriever can return relevant but insufficient context")
print("  (e.g., only 1 of 3 needed chunks).")

---

## Exam Tips

| # | Tip | Why It Matters |
|---|-----|----------------|
| 1 | Precision@K and Recall@K use **binary** relevance (yes/no) | Distinguish from nDCG which uses graded relevance |
| 2 | nDCG is the only standard metric that uses **graded** relevance (0, 1, 2, 3) | If the exam says "graded relevance", the answer is nDCG |
| 3 | MRR only cares about the **first** relevant result | Use MRR when a single good result is enough |
| 4 | Context relevancy != context sufficiency | Relevancy = on topic; sufficiency = enough info to answer |
| 5 | MAP@K is the mean of AP@K across all queries | MAP rewards retrievers that consistently rank relevant docs high |

---

## Key Takeaways

### Binary Relevance Metrics
- **Precision@K**: What fraction of retrieved docs is relevant? High precision = low noise.
- **Recall@K**: What fraction of ALL relevant docs did we find? High recall = good coverage.
- **MRR**: How quickly do we find the first relevant doc? High MRR = relevant results appear early.
- **MAP@K**: Average precision at each relevant position. Rewards consistent quality.

### Graded Relevance Metrics
- **nDCG@K**: Are the MOST relevant docs ranked highest? Uses graded scores (0, 1, 2, 3).
- nDCG normalizes by the ideal ranking, so 1.0 = perfect ordering.

### RAG-Specific Metrics
- **Context relevancy**: Is the retrieved context on-topic for the query?
- **Context sufficiency**: Does the context contain enough information to answer fully?
- These are typically evaluated by an LLM judge, not by a formula.

### Practical Guidance
- Always evaluate on a **golden dataset** with known relevant documents.
- Use multiple metrics -- no single metric tells the full story.
- Low precision? Try re-ranking (next lab). Low recall? Try better embeddings or more chunks.

---

## Next Lab

**Lab 02-05: Re-ranking** -- Your retriever has decent recall but noisy precision. A
re-ranker takes the rough candidates from vector search and re-scores them with a more
accurate model. Two-stage retrieval: fast-but-rough, then slow-but-precise.